In [7]:
import torch
import sys
sys.path.insert(0, "/root/vk_work")
import torch.nn.functional as F
import torchattacks
from torch.utils.data import Subset, DataLoader
from torch import nn

from src.model import CustomResNet

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class Clf(nn.Module):
    def __init__(self, model, simkin=True):
        super().__init__()
        self.model, self.simkin = model, simkin
        self.register_buffer("mean", MEAN)
        self.register_buffer("std", STD)
    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.model(x, mode='clas') if self.simkin else self.model(x)

In [8]:
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset
from src.custom_datasets import STL10RGBDataset

t = transforms.ToTensor()
test = STL10RGBDataset(load_dataset("jxie/stl10")["test"], transform=t)
test_loader = DataLoader(test, batch_size=128, shuffle=False, num_workers=8)

In [9]:
def load(ckpt):
    m = CustomResNet().to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    return Clf(m, simkin=True).to(device).eval()

def noise_load(ckpt):
    m = CustomResNet(noise_sigma=0.5).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    return Clf(m, simkin=True).to(device).eval()

best = load("../models/ResNetSIMv2-30_best.pth")
mid = load("../models/ResNetSIMv2-15_best.pth")
froz = load("../models/ResNetSIMv2_frozen_best.pth")
noisy = noise_load('../models/ResNetSIM_frozen_noisy_best.pth')

/root/vk_work/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/root/vk_work/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [10]:
sub = Subset(test, range(1000))
sub_loader = DataLoader(sub, batch_size=128, shuffle=False, num_workers=8)

def adv_acc(clf, loader, atk=None):
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if atk is not None:
            x = atk(x, y)
        with torch.no_grad():
            correct += (clf(x).argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

for name, clf in [("frozen_noisy", noisy), ("frozen", froz), ("pretrain15", mid), ("pretrain30", best)]:
    print(f"\n{name}: clean {adv_acc(clf, sub_loader):.3f}")
    for eps in [1/255, 2/255, 4/255, 8/255]:
        f = adv_acc(clf, sub_loader, torchattacks.FGSM(clf, eps=eps))
        p = adv_acc(clf, sub_loader, torchattacks.PGD(
            clf, eps=eps, alpha=eps/4, steps=10, random_start=True))
        print(f"  eps={eps*255:.0f}/255  FGSM {f:.3f}  PGD {p:.3f}")


frozen_noisy: clean 0.879
  eps=1/255  FGSM 0.748  PGD 0.708
  eps=2/255  FGSM 0.605  PGD 0.473
  eps=4/255  FGSM 0.360  PGD 0.130
  eps=8/255  FGSM 0.160  PGD 0.001

frozen: clean 0.909
  eps=1/255  FGSM 0.720  PGD 0.672
  eps=2/255  FGSM 0.524  PGD 0.359
  eps=4/255  FGSM 0.268  PGD 0.064
  eps=8/255  FGSM 0.105  PGD 0.000

pretrain15: clean 0.935
  eps=1/255  FGSM 0.674  PGD 0.568
  eps=2/255  FGSM 0.433  PGD 0.182
  eps=4/255  FGSM 0.205  PGD 0.003
  eps=8/255  FGSM 0.083  PGD 0.000

pretrain30: clean 0.932
  eps=1/255  FGSM 0.720  PGD 0.636
  eps=2/255  FGSM 0.486  PGD 0.247
  eps=4/255  FGSM 0.253  PGD 0.011
  eps=8/255  FGSM 0.090  PGD 0.000


In [ ]:
def square_acc(clf, loader, eps, n_queries=1000):
    atk = torchattacks.Square(clf, norm='Linf', eps=eps,
                              n_queries=n_queries, n_restarts=1)
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        x_adv = atk(x, y)
        with torch.no_grad():
            correct += (clf(x_adv).argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

sub = Subset(test, range(400))
aa_loader = DataLoader(sub, batch_size=128, shuffle=False, num_workers=8)

for name, clf in [("freeze", froz), ("freeze+noise", noisy)]:
    for eps in [2/255, 4/255]:
        print(f"{name} Square eps={eps*255:.0f}/255: {square_acc(clf, aa_loader, eps):.3f}")

freeze Square eps=2/255: 0.807
freeze Square eps=4/255: 0.677
freeze+noise Square eps=2/255: 0.900
freeze+noise Square eps=4/255: 0.887


In [ ]:
sub = Subset(test, range(400))
aa_loader = DataLoader(sub, batch_size=128, shuffle=False, num_workers=8)

def eot_pgd(clf, x, y, eps, alpha, steps=20, eot=10):
    x0 = x.clone().detach()
    x_adv = (x0 + torch.empty_like(x0).uniform_(-eps, eps)).clamp(0, 1)
    for _ in range(steps):
        x_adv.requires_grad_(True)
        grad = torch.zeros_like(x_adv)
        for _ in range(eot):
            loss = F.cross_entropy(clf(x_adv), y)
            grad += torch.autograd.grad(loss, x_adv)[0]
        x_adv = x_adv.detach() + alpha * (grad / eot).sign()
        x_adv = torch.min(torch.max(x_adv, x0 - eps), x0 + eps).clamp(0, 1)
    return x_adv.detach()

def eot_acc(clf, loader, eps, reps=5):
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        x_adv = eot_pgd(clf, x, y, eps, alpha=eps/4)
        with torch.no_grad():
            logits = sum(clf(x_adv) for _ in range(reps)) / reps
            correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total


eot_acc(noisy, aa_loader, 2/255)

0.445